<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 6 · Retropropagación desde cero y entrenamiento con PyTorch

**Unidad 6**

PyTorch calcula gradientes con una sola línea, `perdida.backward()`. En este laboratorio vas a hacer a mano lo que esa línea esconde. En la **parte A** programarás en NumPy las piezas de una red neuronal (una capa lineal, la activación ReLU y la salida *softmax* con entropía cruzada), cada una con su pasada **hacia adelante** y su pasada **hacia atrás**; comprobarás los gradientes con diferencias finitas y, uniendo las piezas, entrenarás un perceptrón multicapa con descenso por gradiente estocástico. En la **parte B** pasarás a PyTorch: escribirás el ciclo de entrenamiento de una época y una red convolucional pequeña con una arquitectura dada.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** En la parte A usa solo NumPy (y `scipy.special.logsumexp` si quieres); PyTorch y scikit-learn se usan únicamente para comparar. Las operaciones sobre los datos deben ser vectorizadas: los únicos ciclos permitidos son el de las épocas y minilotes del entrenamiento y el de las entradas en la verificación por diferencias finitas. En la parte B escribe los ciclos de entrenamiento tú mismo, sin bibliotecas de alto nivel.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`. Toda la libreta corre en CPU en menos de un minuto: no hace falta GPU.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "rediseno-2027"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u06-lab06-backprop"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py', 'p8.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/06-redes-neuronales/laboratorios/tests/u06-lab06-backprop/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import copy
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import rdp

rdp.estilo.usar()
torch.manual_seed(2011)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Los datos y el problema

Usaremos los **dígitos escritos a mano** que ya conoces de la Unidad 4 (*Optical Recognition of Handwritten Digits*, E. Alpaydin y C. Kaynak, repositorio UCI, 1998; viene incluido en scikit-learn como `load_digits`). Son 1,797 imágenes de 8×8 de los dígitos 0 a 9, escritos por 43 personas. Cada imagen original era un mapa de bits de 32×32; se dividió en bloques de 4×4 y en cada bloque se contó cuántos píxeles estaban encendidos, de modo que cada uno de los 64 píxeles es un entero entre 0 y 16. Lo dividimos entre 16 para que quede entre 0 y 1.

**El problema.** Reconocer el dígito a partir de los 64 píxeles: clasificación en 10 clases. Usamos la misma partición estratificada que en la Unidad 4 (25 % para prueba, `random_state=2011`). Como referencia, la regresión logística multinomial con `C=10` acierta en 0.971 de las imágenes de prueba. ¿Puede una red neuronal hecha a mano hacerlo mejor?

In [ ]:
digitos = load_digits()
X_todo = digitos.data / 16.0
y_todo = digitos.target

X_ent, X_pru, y_ent, y_pru = train_test_split(X_todo, y_todo, test_size=0.25, stratify=y_todo, random_state=2011)
print(f"entrenamiento: {X_ent.shape} · prueba: {X_pru.shape}")

logistica = LogisticRegression(C=10, max_iter=5000).fit(X_ent, y_ent)
print(f"logística (C = 10), exactitud en prueba: {logistica.score(X_pru, y_pru):.3f}")

fig, ejes = plt.subplots(2, 10, figsize=(11, 3.2), layout="constrained")
for ax, imagen, etiqueta in zip(ejes.ravel(), X_ent[:20], y_ent[:20]):
    ax.imshow(imagen.reshape(8, 8), cmap="gray_r")
    ax.set_title(str(etiqueta))
    ax.axis("off")
fig.suptitle("Veinte imágenes de entrenamiento con su etiqueta");

# Parte A · Una red neuronal en NumPy

## Convención: renglones son ejemplos

En clase escribimos la red para **un** ejemplo, con vectores columna: $\mathbf z = W\mathbf a + \mathbf b$. En el código procesamos un **minilote** de $N$ ejemplos a la vez y conviene que cada renglón sea un ejemplo, como en scikit-learn. Una capa lineal recibe $X$ de $N \times d$ y produce

$$Z = X W + \mathbf{1}\mathbf b^\top ,$$

con $W$ de $d \times m$ y $\mathbf b$ de longitud $m$ (en NumPy, `X @ W + b`: la suma de `b` se «difunde» a todos los renglones). Es la misma capa de clase, transpuesta.

## 1. La capa lineal, hacia adelante y hacia atrás

La retropropagación funciona por piezas. Cada pieza recibe el gradiente de la pérdida $E$ respecto a su **salida**, $\partial E/\partial Z$ (que llamaremos `dZ`, de la misma forma que $Z$), y devuelve el gradiente respecto a su **entrada** y a sus **parámetros**. Para la capa lineal, por la regla de la cadena,

$$\frac{\partial E}{\partial X} = \frac{\partial E}{\partial Z}\, W^\top, \qquad \frac{\partial E}{\partial W} = X^\top \frac{\partial E}{\partial Z}, \qquad \frac{\partial E}{\partial \mathbf b} = \sum_{n=1}^{N} \Bigl(\frac{\partial E}{\partial Z}\Bigr)_{n,:} .$$

Una forma de recordarlas: cada gradiente tiene la forma de lo que se deriva, y solo hay una manera de multiplicar las matrices disponibles para que las dimensiones cuadren. Para convencerte de la segunda, escribe $Z_{nk} = \sum_j X_{nj}W_{jk} + b_k$: el peso $W_{jk}$ aparece en todos los $Z_{nk}$ con $n = 1, \dots, N$, multiplicado por $X_{nj}$, así que $\partial E/\partial W_{jk} = \sum_n X_{nj}\,(\partial E/\partial Z)_{nk}$. La suma sobre $n$ aparece porque el **mismo** peso se usa con todos los ejemplos del minilote.

**Pregunta 1.** Escribe `lineal_adelante(X, W, b)`, que devuelve $Z$, y `lineal_atras(dZ, X, W)`, que devuelve la tupla `(dX, dW, db)` con las fórmulas de arriba. Sin ciclos.

In [ ]:
def lineal_adelante(X, W, b):
    """Capa lineal: Z = X W + b, con X de N × d, W de d × m y b de longitud m."""
    ...


def lineal_atras(dZ, X, W):
    """Gradientes de la capa lineal a partir de dZ = ∂E/∂Z. Devuelve (dX, dW, db)."""
    ...


_X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
_W = np.array([[1.0, 0.0, -1.0], [2.0, 1.0, 0.0]])
_b = np.array([0.5, 0.0, -0.5])
lineal_adelante(_X, _W, _b)

In [ ]:
grader.check("p1")

## 2. La activación ReLU

La función $\operatorname{ReLU}(z) = \max(0, z)$ se aplica entrada por entrada, así que su jacobiana es diagonal y la pasada hacia atrás es un producto entrada por entrada (el $\odot$ de clase):

$$\frac{\partial E}{\partial Z} = \frac{\partial E}{\partial A} \odot \mathbb 1[Z > 0], \qquad A = \operatorname{ReLU}(Z).$$

En $z = 0$ la ReLU no es derivable; por convención (la misma de PyTorch) usamos derivada 0 ahí.

**Pregunta 2.** Escribe `relu(Z)` y `relu_atras(dA, Z)`, que devuelve $\partial E/\partial Z$ a partir de $\partial E/\partial A$ y de la **entrada** $Z$ de la activación.

In [ ]:
def relu(Z):
    """max(0, z) entrada por entrada."""
    ...


def relu_atras(dA, Z):
    """Gradiente de la ReLU: dA donde Z > 0, cero en otro caso."""
    ...


relu(np.array([-2.0, 0.0, 3.0])), relu_atras(np.array([5.0, 5.0, 5.0]), np.array([-2.0, 0.0, 3.0]))

In [ ]:
grader.check("p2")

## 3. *Softmax* y entropía cruzada, estables

La última capa produce, para cada ejemplo, un vector de $K = 10$ **puntajes** (*logits*) $\mathbf z_n$. La *softmax* los convierte en probabilidades y la entropía cruzada mide qué tan poca probabilidad le dio el modelo a la clase correcta $y_n$:

$$\hat p_{nk} = \frac{e^{z_{nk}}}{\sum_{j} e^{z_{nj}}}, \qquad E = -\frac1N\sum_{n=1}^{N} \log \hat p_{n, y_n} = \frac1N\sum_{n=1}^{N}\Bigl[\log\sum_{j} e^{z_{nj}} - z_{n, y_n}\Bigr].$$

La segunda forma es la que se programa: el término $\log\sum_j e^{z_{nj}}$ se calcula con el truco **log-suma-exp** que usamos en el laboratorio 4 (restar el máximo de cada renglón antes de exponenciar), y así nunca se desborda ni se toma $\log 0$.

El gradiente es sorprendentemente simple. Derivando la segunda forma respecto a $z_{nk}$, el primer término da $\hat p_{nk}$ y el segundo da $\mathbb 1[k = y_n]$:

$$\frac{\partial E}{\partial Z} = \frac{1}{N}\bigl(\hat P - Y\bigr),$$

donde $Y$ es la matriz *one-hot* de las etiquetas ($Y_{nk} = 1$ si $k = y_n$). Es el $\delta^{(L)} = \hat{\mathbf y} - \mathbf y$ de clase, dividido entre $N$ porque la pérdida es un **promedio**.

**Pregunta 3.** Escribe `softmax_entropia(Z, y)`, que recibe los puntajes `Z` de $N \times K$ y las etiquetas enteras `y` (de 0 a $K-1$) y devuelve la tupla `(perdida, dZ)`: la entropía cruzada media (un `float`) y su gradiente $\frac1N(\hat P - Y)$. Debe funcionar sin avisos ni `nan` aunque los puntajes valgan miles. Pista: con `Z[np.arange(N), y]` tomas, en cada renglón, la entrada de la clase correcta.

In [ ]:
def softmax_entropia(Z, y):
    """Entropía cruzada media de softmax(Z) con etiquetas enteras y, y su gradiente respecto a Z."""
    ...


softmax_entropia(np.zeros((2, 4)), np.array([0, 3]))

In [ ]:
grader.check("p3")

## 4. Verificar gradientes con diferencias finitas

Un error en la pasada hacia atrás no produce ningún mensaje: la red simplemente aprende peor, o no aprende. Por eso, antes de entrenar, se **verifica** el gradiente comparándolo con una aproximación numérica. La **diferencia centrada**

$$\frac{\partial f}{\partial x_i} \approx \frac{f(\mathbf x + h\mathbf e_i) - f(\mathbf x - h\mathbf e_i)}{2h}$$

tiene error $O(h^2)$ (los términos de orden $h$ del desarrollo de Taylor se cancelan), mucho mejor que el $O(h)$ de la diferencia hacia adelante. Cuesta dos evaluaciones de $f$ por entrada: imposible para entrenar una red de millones de pesos, perfecto para revisar una red chica.

Para comparar dos gradientes no sirve el error absoluto (un gradiente de tamaño $10^{-8}$ y otro de $10^{3}$ no se comparan igual), así que se usa el **error relativo**

$$\operatorname{err}(\mathbf a, \mathbf b) = \frac{\lVert \mathbf a - \mathbf b\rVert}{\max\bigl(\lVert \mathbf a\rVert + \lVert \mathbf b\rVert,\ 10^{-12}\bigr)},$$

con la norma euclidiana de todas las entradas. En doble precisión, un gradiente correcto da errores relativos de $10^{-7}$ o menos; un error de programación suele dar $10^{-2}$ o más.

**Pregunta 4.** Escribe `gradiente_numerico(f, x, h=1e-5)`, que recibe una función `f` que toma un arreglo y devuelve un número, y un arreglo `x` de **cualquier forma**, y devuelve un arreglo de la forma de `x` con la diferencia centrada en cada entrada. Aquí sí puedes recorrer las entradas con un ciclo (por ejemplo con `np.ndindex(x.shape)`). Al terminar, `x` debe quedar **exactamente** como estaba. Escribe también `error_relativo(a, b)`, que devuelve un `float`.

In [ ]:
def gradiente_numerico(f, x, h=1e-5):
    """Gradiente de f en x por diferencias centradas, entrada por entrada."""
    ...


def error_relativo(a, b):
    """‖a − b‖ / max(‖a‖ + ‖b‖, 1e-12)."""
    ...


gradiente_numerico(lambda v: np.sum(v ** 3), np.array([1.0, 2.0, -1.0]))

In [ ]:
grader.check("p4")

## 5. Una red de dos capas: adelante y atrás

Ahora juntamos las piezas en un perceptrón multicapa con una capa oculta de $h$ neuronas ReLU:

$$Z_1 = X W_1 + \mathbf b_1, \quad A_1 = \operatorname{ReLU}(Z_1), \quad Z_2 = A_1 W_2 + \mathbf b_2, \quad E = \text{entropía cruzada}(Z_2, \mathbf y).$$

La pasada hacia atrás recorre el mismo camino al revés: `softmax_entropia` da $\partial E/\partial Z_2$; `lineal_atras` de la segunda capa da los gradientes de $W_2$, $\mathbf b_2$ y $A_1$; `relu_atras` lleva $\partial E/\partial A_1$ a $\partial E/\partial Z_1$; y `lineal_atras` de la primera capa da los de $W_1$ y $\mathbf b_1$. Por eso hay que **guardar** $X$, $Z_1$ y $A_1$ durante la pasada hacia adelante: la pasada hacia atrás los necesita.

Los parámetros se guardan en un diccionario con llaves `"W1"`, `"b1"`, `"W2"`, `"b2"`. La función de abajo los inicializa con la regla de **He**: pesos normales con varianza $2/n_{\text{entrada}}$, la que mantiene estable la varianza de las activaciones capa tras capa con ReLU, y sesgos en cero.

In [ ]:
def iniciar_parametros(d, h, k, semilla=2011):
    """Perceptrón de d entradas, h neuronas ocultas y k clases, con inicialización de He."""
    rng = np.random.default_rng(semilla)
    return {
        "W1": rng.normal(0.0, np.sqrt(2.0 / d), size=(d, h)),
        "b1": np.zeros(h),
        "W2": rng.normal(0.0, np.sqrt(2.0 / h), size=(h, k)),
        "b2": np.zeros(k),
    }


def predecir(params, X):
    """Clase más probable para cada renglón de X."""
    Z2 = lineal_adelante(relu(lineal_adelante(X, params["W1"], params["b1"])), params["W2"], params["b2"])
    return np.argmax(Z2, axis=1)

**Pregunta 5.** Escribe `adelante_atras(params, X, y)`, que devuelve la tupla `(perdida, grads)`: la entropía cruzada media de la red en los datos `X`, `y` y un diccionario `grads` con las mismas llaves y formas que `params`. Usa tus funciones de las preguntas 1 a 3.

In [ ]:
def adelante_atras(params, X, y):
    """Pérdida de la red de dos capas y gradientes respecto a todos sus parámetros."""
    ...


params = iniciar_parametros(64, 32, 10)
perdida, grads = adelante_atras(params, X_ent[:50], y_ent[:50])
print(f"pérdida inicial: {perdida:.4f}  (log 10 = {np.log(10):.4f})")

Verifiquemos cada gradiente con tus diferencias finitas, en una red pequeña con pocos ejemplos (para que sea rápido). Si todo está bien, los cuatro errores relativos deben ser del orden de $10^{-8}$ o menores.

In [ ]:
_p = iniciar_parametros(64, 8, 10, semilla=1)
_Xv, _yv = X_ent[:20], y_ent[:20]
_, _g = adelante_atras(_p, _Xv, _yv)
for llave in ["W1", "b1", "W2", "b2"]:
    numerico = gradiente_numerico(lambda v: adelante_atras({**_p, llave: v}, _Xv, _yv)[0], _p[llave].copy())
    print(f"{llave}: error relativo {error_relativo(_g[llave], numerico):.2e}")

In [ ]:
grader.check("p5")

## 6. Descenso por gradiente estocástico

Con el gradiente en la mano, entrenar es repetir $\theta \leftarrow \theta - \eta\,\nabla E_{\text{lote}}(\theta)$ para cada parámetro. En cada **época** se revuelven los ejemplos de entrenamiento y se recorren en **minilotes** de `tam_lote` ejemplos consecutivos de esa permutación (el último puede ser más chico); por cada minilote se da un paso.

**Pregunta 6.** Escribe `entrenar_sgd(params, X, y, tasa=0.1, epocas=30, tam_lote=32, semilla=2011)`, que entrena la red con descenso por gradiente estocástico y devuelve la tupla `(params, historia)`: los parámetros entrenados (en un diccionario; puedes actualizar el original o una copia) e `historia`, una lista con la pérdida en **todo** el conjunto `X`, `y` al final de cada época. Revuelve con `rng = np.random.default_rng(semilla)` y `rng.permutation(N)` al inicio de cada época.

In [ ]:
def entrenar_sgd(params, X, y, tasa=0.1, epocas=30, tam_lote=32, semilla=2011):
    """Entrena la red de dos capas con SGD por minilotes. Devuelve (params, historia)."""
    ...


inicio = time.perf_counter()
red, historia = entrenar_sgd(iniciar_parametros(64, 64, 10), X_ent, y_ent, tasa=0.1, epocas=30, tam_lote=32)
exactitud_np = float(np.mean(predecir(red, X_pru) == y_pru))
print(f"tiempo: {time.perf_counter() - inicio:.2f} s · exactitud en prueba: {exactitud_np:.3f}")

In [ ]:
grader.check("p6")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(range(1, len(historia) + 1), historia, color=rdp.estilo.AZUL, marker="o")
ax.set_title("Entrenamiento de la red de dos capas en NumPy")
ax.set_xlabel("época")
ax.set_ylabel("entropía cruzada en entrenamiento (escala log)");

Una red de 64 neuronas ocultas, escrita en unas 30 líneas de NumPy, ya iguala o supera a la logística de la Unidad 4. Los saltos ocasionales de la curva son el precio del descenso **estocástico**: algunos minilotes empujan los pesos en una dirección mala para el conjunto completo.

<!-- BEGIN QUESTION -->

**Reflexión 1 · Simetría.** ¿Qué pasa si se inicializan **todos** los pesos y sesgos en cero? Pruébalo: entrena con `{k: np.zeros_like(v) for k, v in iniciar_parametros(64, 64, 10).items()}` y mide la exactitud en prueba; usa tu `adelante_atras` para ver qué gradientes son cero y por qué (sigue las fórmulas de las preguntas 1 y 2). Luego prueba con todos los pesos iguales a una constante pequeña distinta de cero (por ejemplo 0.01): ¿las 64 neuronas ocultas aprenden cosas distintas? Demuestra, por inducción sobre los pasos de SGD, que si dos neuronas ocultas empiezan con los mismos pesos de entrada y de salida, siguen iguales para siempre. ¿Por qué la inicialización aleatoria rompe ese problema, y por qué la escala $2/n_{\text{entrada}}$ importa además del azar?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2 · ¿Qué tan chico debe ser $h$?** Con la red pequeña de la pregunta 5 (`iniciar_parametros(64, 8, 10, semilla=1)` y los primeros 20 ejemplos), calcula el error relativo entre tu gradiente de `W2` y el numérico para $h = 10^{-1}, 10^{-2}, \dots, 10^{-12}$ y grafícalo en escala log-log. Explica la forma de la curva: ¿por qué el error baja al principio como $h^2$, y por qué vuelve a **subir** cuando $h$ es muy chico? (Pista: ¿cuántas cifras significativas tiene un `float64` y qué pasa al restar dos números casi iguales?) Estima el $h$ óptimo con un argumento de balance entre los dos errores. Por último: la ReLU no es derivable en 0; ¿cuándo podría eso hacer fallar la verificación aunque tu código esté bien?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

# Parte B · Lo mismo, y más, con PyTorch

En PyTorch no hace falta escribir la pasada hacia atrás: cada operación sobre tensores con `requires_grad=True` se anota en un **grafo de cómputo**, y `perdida.backward()` lo recorre al revés aplicando, pieza por pieza, lo que tú programaste en la parte A (diferenciación automática en modo inverso). Lo que sí hay que escribir es el **ciclo de entrenamiento**. Con un `DataLoader`, que entrega los datos en minilotes revueltos, una época consiste en, para cada minilote:

1. poner en cero los gradientes acumulados (`optimizador.zero_grad()`), porque PyTorch los **suma** en cada `backward`;
2. calcular las predicciones y la pérdida;
3. `perdida.backward()` para obtener los gradientes;
4. `optimizador.step()` para actualizar los parámetros.

Antes del ciclo se llama `modelo.train()`, que activa el comportamiento de entrenamiento de capas como *dropout* o la normalización por lotes; para evaluar se usa `modelo.eval()` y `torch.no_grad()` (abajo está `evaluar`, ya escrita).

In [ ]:
def tensores(X, y):
    """Convierte las imágenes (N × 64) en un tensor de N × 1 × 8 × 8 y las etiquetas en enteros."""
    return torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 8, 8), torch.tensor(y, dtype=torch.long)


datos_ent = TensorDataset(*tensores(X_ent, y_ent))
datos_pru = TensorDataset(*tensores(X_pru, y_pru))
cargador_pru = DataLoader(datos_pru, batch_size=256)


def evaluar(modelo, cargador):
    """Exactitud del modelo en los datos del cargador."""
    modelo.eval()
    aciertos, total = 0, 0
    with torch.no_grad():
        for xb, yb in cargador:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            aciertos += (modelo(xb).argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return aciertos / total

**Pregunta 7.** Escribe `entrenar_epoca(modelo, cargador, optimizador, perdida)`, que pone el modelo en modo de entrenamiento, recorre una vez el `cargador` dando un paso del `optimizador` por minilote (mueve cada minilote a `dispositivo`) y devuelve la **pérdida media por ejemplo** de la época como `float`. `perdida` es una función como `F.cross_entropy` que devuelve el promedio del minilote; como el último minilote puede ser más chico, pondera cada promedio por el tamaño de su minilote: $\sum_b N_b\,E_b \big/ \sum_b N_b$. Usa `.item()` para sacar el número de un tensor.

In [ ]:
def entrenar_epoca(modelo, cargador, optimizador, perdida):
    """Una época de entrenamiento. Devuelve la pérdida media por ejemplo."""
    ...


# El mismo perceptrón de la parte A (64 → 64 → 10), ahora en PyTorch
_ = torch.manual_seed(2011)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)).to(dispositivo)
optimizador = torch.optim.SGD(mlp.parameters(), lr=0.1)
cargador_ent = DataLoader(datos_ent, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(2011))
for epoca in range(30):
    perdida_mlp = entrenar_epoca(mlp, cargador_ent, optimizador, F.cross_entropy)
exactitud_mlp = evaluar(mlp, cargador_pru)
print(f"MLP en PyTorch · pérdida de la última época: {perdida_mlp:.4f} · exactitud en prueba: {exactitud_mlp:.3f}")

In [ ]:
grader.check("p7")

## 8. Una red convolucional pequeña

El perceptrón trata los 64 píxeles como una lista sin orden: si revolviéramos los píxeles de todas las imágenes de la misma manera, aprendería exactamente igual. Una red **convolucional** aprovecha que los píxeles vecinos están relacionados: cada filtro de 3×3 recorre la imagen buscando el mismo patrón local en todas partes (pesos **compartidos**), y el *pooling* reduce la resolución a la mitad. Construye esta arquitectura, para entradas de $1 \times 8 \times 8$:

| capa | salida |
|---|---|
| convolución de 1 a 8 canales, filtro 3×3, `padding=1` | 8 × 8 × 8 |
| ReLU y *max pooling* de 2×2 | 8 × 4 × 4 |
| convolución de 8 a 16 canales, filtro 3×3, `padding=1` | 16 × 4 × 4 |
| ReLU y *max pooling* de 2×2 | 16 × 2 × 2 |
| aplanar | 64 |
| capa lineal de 64 a 10 | 10 puntajes |

La salida son los **puntajes** (sin *softmax*: `F.cross_entropy` la aplica por dentro, de forma estable, como tu `softmax_entropia`).

**Pregunta 8.** Completa la clase `RedConvolucional` (las capas con parámetros se crean en `__init__` y se usan en `forward`; para ReLU y *pooling* puedes usar `F.relu` y `F.max_pool2d`). Después, en la celda que sigue, calcula a mano el número de parámetros de la red y escríbelo en `num_parametros` como un entero (no lo calcules con código: la celda de abajo lo compara con el conteo de PyTorch). Recuerda que una convolución de $c_{\text{ent}}$ a $c_{\text{sal}}$ canales con filtros de $3 \times 3$ tiene $c_{\text{sal}}(9\,c_{\text{ent}} + 1)$ parámetros.

In [ ]:
class RedConvolucional(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...


num_parametros = ...

_ = torch.manual_seed(2011)
cnn = RedConvolucional().to(dispositivo)
print(f"parámetros (PyTorch): {sum(p.numel() for p in cnn.parameters()):,} · tu cuenta: {num_parametros}")
print("forma de la salida para 5 imágenes:", tuple(cnn(torch.zeros(5, 1, 8, 8, device=dispositivo)).shape))

In [ ]:
grader.check("p8")

Ahora la entrenamos con **Adam** y tu `entrenar_epoca`, 20 épocas.

In [ ]:
_ = torch.manual_seed(2011)
cnn = RedConvolucional().to(dispositivo)
optimizador = torch.optim.Adam(cnn.parameters(), lr=0.01)
cargador_ent = DataLoader(datos_ent, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(2011))
inicio = time.perf_counter()
for epoca in range(20):
    perdida_cnn = entrenar_epoca(cnn, cargador_ent, optimizador, F.cross_entropy)
exactitud_cnn = evaluar(cnn, cargador_pru)
print(f"tiempo: {time.perf_counter() - inicio:.1f} s · pérdida de la última época: {perdida_cnn:.4f}")

print(f"\n{'modelo':<34}{'parámetros':>11}{'exactitud':>11}")
print(f"{'logística (U4, C = 10)':<34}{650:>11,}{logistica.score(X_pru, y_pru):>11.3f}")
print(f"{'MLP en NumPy (64 ocultas)':<34}{64 * 64 + 64 + 64 * 10 + 10:>11,}{exactitud_np:>11.3f}")
print(f"{'MLP en PyTorch (64 ocultas)':<34}{sum(p.numel() for p in mlp.parameters()):>11,}{exactitud_mlp:>11.3f}")
print(f"{'red convolucional':<34}{sum(p.numel() for p in cnn.parameters()):>11,}{exactitud_cnn:>11.3f}")

La red convolucional usa menos de la mitad de los parámetros del perceptrón y acierta igual o más. (El MLP de PyTorch no da exactamente lo mismo que el tuyo: PyTorch inicializa los pesos con otra regla, uniforme y de varianza menor que la de He, y revuelve los datos con otro generador.)

Abajo están los ocho filtros de la primera capa. Cada uno responde a un patrón local de 3×3, aprendido de los datos y no diseñado a mano; varios se parecen a detectores de bordes en alguna dirección. Con imágenes de 8×8 y tan pocos datos no hay que sobreinterpretarlos.

In [ ]:
filtros = [p for p in cnn.parameters() if p.dim() == 4][0].detach().cpu().numpy()[:, 0]  # pesos de la primera convolución
escala_f = np.abs(filtros).max()
fig, ejes = plt.subplots(1, 8, figsize=(11, 2.2), layout="constrained")
for i, (ax, filtro) in enumerate(zip(ejes, filtros)):
    ax.imshow(filtro, cmap="RdBu_r", vmin=-escala_f, vmax=escala_f)
    ax.set_title(f"filtro {i + 1}")
    ax.axis("off")
fig.suptitle("Filtros de 3×3 de la primera capa convolucional (rojo: peso positivo; azul: negativo)");

<!-- BEGIN QUESTION -->

**Reflexión 3.** Revuelve los 64 píxeles de **todas** las imágenes con una misma permutación fija (`perm = np.random.default_rng(0).permutation(64)`; usa `X_ent[:, perm]` y `X_pru[:, perm]`) y vuelve a entrenar la red de NumPy y la red convolucional con los mismos hiperparámetros. ¿Qué le pasa a la exactitud de cada una? Explica el resultado en términos de lo que cada arquitectura supone sobre los datos (su sesgo inductivo).

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()